In [1]:
import numpy as np
from tqdm.auto import tqdm

from embedder import Embedder
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import VectorSearch, Index

embed = Embedder()

2026-06-29 10:12:36.174719449 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


Q1

In [7]:
v = embed.encode("How does approximate nearest neighbor search work?")
print(f"len(v) = {len(v)}")
print(f"v[0] = {v[0]:.4f}")

len(v) = 384
v[0] = -0.0206


In [6]:
v[0]

np.float64(-0.02058203437252893)

Q2

In [8]:
reader = GithubRepositoryDataReader(  # создаём ридер для чтения файлов из GitHub репозитория
    repo_owner="DataTalksClub",       # владелец репозитория (организация или юзер)
    repo_name="llm-zoomcamp",         # название репозитория
    commit_id="8c1834d",              # конкретный коммит — фиксируем версию, чтобы результаты были воспроизводимы
    allowed_extensions={"md"},        # читаем только .md файлы (Markdown документация)
    filename_filter=lambda path: "/lessons/" in path,  # фильтр: берём только файлы из папки /lessons/
)
files = reader.read()  # скачиваем/читаем файлы из репо по заданным параметрам

documents = [file.parse() for file in files]  # парсим каждый файл в словарь {'filename', 'content'}
documents[0]  # смотрим на первый документ — проверка что всё прочиталось корректно

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [9]:
len(documents)

72

In [10]:
target_name = "02-vector-search/lessons/07-sqlitesearch-vector.md"
target_doc = next(d for d in documents if d["filename"] == target_name)

In [13]:
[d for d in documents if d["filename"] == target_name]

[{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done

In [14]:
next(d for d in documents if d["filename"] == target_name)

{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done 

In [12]:
target_doc

{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done 

In [16]:
doc_vec = embed.encode(target_doc["content"])
cos = float(doc_vec.dot(v))
print(f"cosine similarity = {cos:.4f}")

cosine similarity = 0.3611


Q3

In [17]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [21]:
chunks[1]

{'start': 1000,
 'content': 'the next\nword based on what you typed so far.\n\nA large language model does the same thing, but at a much larger scale.\nIt has billions of parameters and is trained on most of the text on the\ninternet. When it predicts the next word, it feels like you\'re talking\nto an intelligent being. It understands what you ask and gives\nmeaningful answers.\n\nIn this course, we treat LLMs as black boxes. We won\'t look inside or\ncover the theory, and we won\'t host a model ourselves. We use an LLM\nprovider and call it over an API. For us, an LLM is a box: text goes in,\ntext comes out.\n\nBut LLMs have limitations:\n\n- Knowledge cutoff: they only know what was in their training data.\n  If you ask about something that happened after training, they won\'t\n  know - or worse, they\'ll make something up.\n- No access to your data: they can\'t see your documents, databases,\n  or internal systems unless you provide that information.\n- Hallucinations: they sometim

In [22]:
chunk_texts = [c["content"] for c in chunks]
batch_size = 50
X = []
for i in tqdm(range(0, len(chunk_texts), batch_size)):
    X.extend(embed.encode_batch(chunk_texts[i:i + batch_size]))
X = np.array(X)

  0%|          | 0/6 [00:00<?, ?it/s]

In [24]:
len(chunk_texts)

295

In [27]:
X.shape

(295, 384)

In [ ]:
scores = X.dot(v)

array([ 3.15187611e-01,  2.01479664e-01,  5.90559037e-02, -7.67661288e-02,
        1.18452466e-01, -1.41697012e-01, -2.81406495e-02, -4.65669117e-02,
       -2.06994543e-02, -6.07744147e-02,  2.13273769e-01,  8.87600958e-02,
       -1.97269268e-02,  3.11629985e-01,  5.51034635e-01,  2.04008152e-01,
        2.12515802e-01,  1.93649107e-01,  2.51961267e-01,  1.31078610e-01,
        2.59120607e-01,  7.63816369e-02,  9.59193203e-02,  9.81471228e-03,
       -3.59107168e-02,  1.01211406e-02,  1.10786940e-01, -9.90259915e-02,
       -3.71170645e-02,  7.59057333e-02, -3.35340234e-02,  8.86841484e-03,
        1.02636448e-01,  6.89615413e-02,  1.29408854e-01,  2.57709121e-01,
        3.23680576e-01,  1.06350076e-01,  5.61891208e-02,  2.34017441e-01,
        1.97954358e-01,  9.64296342e-02,  1.93709934e-01,  2.16719269e-01,
        3.48340450e-01,  5.10906541e-02,  2.05212833e-01,  1.05416182e-01,
       -3.25432660e-02,  4.94665347e-02,  2.38574873e-01, -3.44207304e-02,
        1.82165430e-01,  

In [34]:
scores[94]


np.float64(0.6489017718578813)

In [30]:
best = int(np.argmax(scores))
best

94

In [31]:
chunks[best]

{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

In [33]:
print(f"top chunk filename = {chunks[best]['filename']} (score={scores[best]:.4f})")

top chunk filename = 02-vector-search/lessons/07-sqlitesearch-vector.md (score=0.6489)


Q4

In [37]:
X[5].shape

(384,)

In [39]:
chunks[5]

{'start': 2000,
 'content': '\nThe safest way to store the key is in a `.env` file that never gets\ncommitted to git.\n\nCreate a `.env` file in your project folder and put your API key in\nit:\n\n```bash\nOPENAI_API_KEY=sk-YOUR_KEY_HERE\n```\n\nNow add `.env` to `.gitignore` to make sure you never accidentally\ncommit your key:\n\n```bash\n.env\n```\n\nNever commit `.env` to git. Treat the API key like a password. If it\nleaks, someone else can run up charges on your account.\n\n## Starting Jupyter\n\nStart Jupyter:\n\n```bash\nuv run jupyter notebook\n```\n\nCreate a new notebook. Throughout the course, you\'ll copy code from\nthe section notes into notebook cells.\n\nCheck that the OpenAI client works:\n\n```python\nfrom dotenv import load_dotenv\nload_dotenv()\n\nfrom openai import OpenAI\nopenai_client = OpenAI()\n```\n\nIf you see an error, make sure the key in your `.env` file is\ncorrect.\n\nFor Groq or other OpenAI-compatible providers, add the key to\n`.env`:\n\n```bash\nGROQ

In [38]:
chunks[5]["content"][:500]  # выводим первые 500 символов текста пятого чанка

'\nThe safest way to store the key is in a `.env` file that never gets\ncommitted to git.\n\nCreate a `.env` file in your project folder and put your API key in\nit:\n\n```bash\nOPENAI_API_KEY=sk-YOUR_KEY_HERE\n```\n\nNow add `.env` to `.gitignore` to make sure you never accidentally\ncommit your key:\n\n```bash\n.env\n```\n\nNever commit `.env` to git. Treat the API key like a password. If it\nleaks, someone else can run up charges on your account.\n\n## Starting Jupyter\n\nStart Jupyter:\n\n```bash\nuv run jupyter noteb'

In [35]:
vindex = VectorSearch()
vindex.fit(X, chunks)

In [40]:
v_q4 = embed.encode("What metric do we use to evaluate a search engine?")
v_q4

array([-3.45251120e-02, -4.76347907e-02, -9.52288143e-02,  2.40308090e-02,
       -1.80039094e-02,  3.22336786e-02,  4.11009181e-04,  2.86892654e-02,
        2.71596053e-02, -6.36475643e-02, -3.86707619e-02, -5.52097101e-02,
        3.84640827e-02,  3.63455416e-02, -9.32710316e-02, -5.29945155e-02,
        8.20461574e-02, -5.03414745e-03,  2.50910438e-02, -8.70656696e-02,
        7.20100754e-02,  1.29639035e-02,  1.16451658e-01, -8.20982256e-02,
       -6.52023347e-03, -4.00227584e-02, -8.36716391e-02, -1.24123525e-02,
       -1.90502106e-02, -6.24077402e-03, -3.73876830e-02,  6.12152591e-03,
        5.60926979e-02,  6.56479763e-02, -6.34850137e-02,  3.43527274e-02,
       -3.35941763e-02, -3.51476658e-02,  2.54281298e-02, -9.75776093e-03,
       -4.84886223e-02, -2.35025387e-02, -2.90664668e-02,  3.65187041e-02,
        2.23322831e-02, -3.29535923e-02, -4.82611746e-02,  8.59540277e-03,
       -3.70969933e-03,  5.49367302e-02, -9.72623872e-02, -1.93188061e-04,
       -1.25472844e-02, -

In [41]:
v_q4.shape

(384,)

In [42]:
vres = vindex.search(v_q4, num_results=5)
print(f"first result filename = {vres[0]['filename']}")

first result filename = 04-evaluation/lessons/05-search-metrics.md


In [43]:
vres

[{'start': 0,
  'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our set